In [ ]:
# ============================================================
# Evaluation Notebook — Analyse & Visualise Load Test Results
# ============================================================
# This notebook reads the CSV log files produced by RunPerfScenario
# and generates summary tables + bar charts so you can compare
# query performance across different scenarios (e.g. Import vs
# DirectQuery, with vs without RLS, etc.).
#
# Two output modes are available (set OUTPUT_MODE below):
#   "A" — aggregate: one summary row per scenario folder
#   "B" — per-user:  one row per (scenario x user), with side-by-side charts
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
import os
import re

spark = SparkSession.builder.getOrCreate()

# ============================================================
# CONFIGURATION
# ============================================================
# Each tuple describes one test-run folder to include in the analysis:
#   (workspace_name, folder_name, display_label)
#
# - workspace_name : the Power BI workspace used during the test
# - folder_name    : the exact subfolder under .../logs/<workspace>/
# - display_label  : short label shown in charts and tables
#
# Add or remove entries to compare as many runs as you like.
folder_specs = [
    ("PBI MRL UAT CRWEG ZEROGRAVITY  SS", "Query DirectQuery CRWEG-Dragos PRU&PKO-20260303_1500", "DQ w RLS"),
    ("MRL GCTO POC", "Query Import RLS 5USR-20260302-123837", "Import with RLS"),
]

# Fraction of initial rows to discard as warm-up (0 = keep all, 0.05 = drop first 5%)
warmup_ratio = 0

# ============================================================
# OUTPUT MODE
# ============================================================
# "A" — All CSV files within the same folder are merged into one measurement
#        per folder.  Good for a quick high-level comparison.
# "B" — Every CSV file (i.e. every virtual user) is shown individually and
#        compared across folders.  Good for spotting per-user variance.
OUTPUT_MODE = "B"

# ============================================================
# SETUP — derive lakehouse paths from the config above
# ============================================================

# Safety filter: keep only folders whose name contains "Import" or "DirectQuery"
folder_specs = [fs for fs in folder_specs if "Import" in fs[1] or "DirectQuery" in fs[1]]

# Full lakehouse paths to each log folder
base_paths = [
    f"/lakehouse/default/Files/PerfScenarios/logs/{ws}/{f}"
    for ws, f, _label in folder_specs
]

# (full_path, display_label) pairs used when loading data
folders = [
    (f"{ws}/{f}", label)
    for ws, f, label in folder_specs
]

# ============================================================
# HELPER FUNCTIONS
# ============================================================

# Pre-compiled regex to extract the user index from filenames like
# "MyTest_tartau_0.csv"  →  group(1) = "0"
_USER_PATTERN = re.compile(r"_user_(\d+)\.csv$")


def discover_user_files(base_path):
    """
    List CSV files in *base_path* and return them sorted by user index.

    If filenames match the pattern ``*_user_<N>.csv`` they are sorted
    numerically by N.  Otherwise all CSVs are returned in alphabetical order.
    """
    all_files = [f for f in os.listdir(base_path) if f.endswith(".csv")]
    matched = [f for f in all_files if _USER_PATTERN.search(f)]
    if matched:
        return sorted(matched, key=lambda x: int(_USER_PATTERN.search(x).group(1)))
    return sorted(all_files)


def load_csv(path):
    """
    Load a single CSV result file and return a clean pandas DataFrame.

    Tries Spark first (better for large files on the lakehouse) and
    falls back to plain pandas if Spark fails.  Rows with missing or
    non-numeric ``duration`` values are dropped.
    """
    try:
        sdf = (
            spark.read
            .option("header", True)
            .option("inferSchema", False)
            .option("multiLine", True)
            .option("quote", '"')
            .option("escape", '"')
            .csv(path)
        )
        pdf = sdf.toPandas()
    except Exception:
        pdf = pd.read_csv(path, quotechar='"', escapechar='\\', on_bad_lines='skip', dtype=str)

    # Clean: keep only rows where duration is a valid number
    pdf = pdf[pdf["duration"].notna() & ~pdf["duration"].isin(["NULL", "null", ""])]
    pdf["duration"] = pd.to_numeric(pdf["duration"], errors="coerce")
    pdf["rows"] = pd.to_numeric(pdf["rows"], errors="coerce")
    return pdf[pdf["duration"].notna()].reset_index(drop=True)


def remove_warmup(df, ratio):
    """
    Drop the first *ratio* fraction of rows to exclude warm-up queries.

    For example, ratio=0.05 removes the first 5% of rows.
    """
    cut = int(len(df) * ratio)
    return df.iloc[cut:].reset_index(drop=True)


def load_folder_data(base_path):
    """
    Load every user CSV from *base_path* into a dict ``{label: DataFrame}``.

    Each file becomes one entry.  The label is ``User_0``, ``User_1``, etc.
    Warm-up rows are removed according to ``warmup_ratio``.
    """
    users = {}
    for i, filename in enumerate(discover_user_files(base_path)):
        match = _USER_PATTERN.search(filename)
        label = f"User_{match.group(1)}" if match else f"User_{i}"
        df = load_csv(f"{base_path}/{filename}")
        df = remove_warmup(df, warmup_ratio)
        users[label] = df
    return users


def summarize(df):
    """
    Compute key statistics for a DataFrame of query results.

    Returns a pandas Series with: query_count, avg, p50, p90, p95, p99,
    max duration, and average row count.
    """
    d = df["duration"].dropna()
    return pd.Series({
        "query_count":  len(df),
        "avg_duration": round(d.mean(), 1),
        "p50_duration": round(d.quantile(0.50), 1),
        "p90_duration": round(d.quantile(0.90), 1),
        "p95_duration": round(d.quantile(0.95), 1),
        "p99_duration": round(d.quantile(0.99), 1),
        "max_duration": round(d.max(), 1),
        "avg_rows":     round(df["rows"].mean(), 1),
    })


def annotate_bars(ax):
    """Add numeric value labels above every bar in a matplotlib/seaborn axes."""
    for p in ax.patches:
        height = p.get_height()
        if not np.isnan(height):
            ax.annotate(
                f"{height:.1f}",
                (p.get_x() + p.get_width() / 2.0, height),
                ha="center", va="bottom", fontsize=9,
            )


# ============================================================
# LOAD DATA — read all CSV files from every configured folder
# ============================================================

records = []
for (folder_path, label), base_path in zip(folders, base_paths):
    users = load_folder_data(base_path)
    for user_label, df in users.items():
        df = df.copy()
        df["dimension"] = label      # tag every row with its scenario label
        df["user"] = user_label      # tag every row with its user label
        records.append(df)

all_data = pd.concat(records, ignore_index=True)
print(f"Rows loaded: {len(all_data)}  |  Output mode: {OUTPUT_MODE}")

# ============================================================
# MODE A — one aggregated summary row per scenario folder
# ============================================================
if OUTPUT_MODE == "A":

    # Compute percentile statistics grouped by scenario
    overall_summary = (
        all_data
        .groupby("dimension")["duration"]
        .agg(
            queries="count",
            avg="mean",
            p50=lambda x: x.quantile(0.50),
            p90=lambda x: x.quantile(0.90),
            p95=lambda x: x.quantile(0.95),
            p99=lambda x: x.quantile(0.99),
            max="max",
        )
        .round(1)
        .reset_index()
    )
    display(overall_summary)

    # Grouped bar chart: one cluster per metric, one bar per scenario
    metrics_to_plot = ["avg", "p50", "p90", "p95", "p99", "max"]
    plot_data = overall_summary.melt(
        id_vars="dimension",
        value_vars=metrics_to_plot,
        var_name="Metric",
        value_name="Duration",
    )

    plt.figure(figsize=(10, 6))
    ax = sns.barplot(data=plot_data, x="Metric", y="Duration", hue="dimension", palette="Set2")
    annotate_bars(ax)
    ax.set_title("Mode A - Overall Query Duration Summary by Folder")
    ax.set_ylabel("Duration (s)")
    ax.set_xlabel("Metric")
    plt.grid(axis="y", alpha=0.3)
    plt.legend(title="Folder")
    plt.tight_layout()
    plt.show()

# ============================================================
# MODE B — one row per (scenario x user), with comparison charts
# ============================================================
elif OUTPUT_MODE == "B":

    # Build per-user summary using an explicit loop to avoid pandas
    # groupby.apply quirks with extra index levels in some versions.
    user_rows = []
    for (dim, usr), grp in all_data.groupby(["dimension", "user"]):
        row = summarize(grp)
        row["dimension"] = dim
        row["user"] = usr
        user_rows.append(row)

    per_user_summary = pd.DataFrame(user_rows)[
        ["dimension", "user", "avg_duration", "p50_duration", "p90_duration",
         "p95_duration", "p99_duration", "max_duration", "query_count"]
    ]
    # Sort numerically by user index so User_2 comes after User_1 (not User_10)
    per_user_summary["_sort"] = per_user_summary["user"].str.extract(r"(\d+)$")[0].astype(int)
    per_user_summary = (
        per_user_summary
        .sort_values(["dimension", "_sort"])
        .drop(columns="_sort")
        .reset_index(drop=True)
    )
    display(per_user_summary)

    # Individual charts: one chart per metric, x-axis = user, hue = scenario
    metrics_to_plot = [
        ("avg_duration", "Avg"),
        ("p50_duration", "p50"),
        ("p90_duration", "p90"),
        ("p95_duration", "p95"),
        ("p99_duration", "p99"),
        ("max_duration", "Max"),
    ]

    for col, title in metrics_to_plot:
        plt.figure(figsize=(max(10, len(per_user_summary) * 0.8), 5))
        ax = sns.barplot(data=per_user_summary, x="user", y=col, hue="dimension", palette="Set2")
        annotate_bars(ax)
        ax.set_title(f"Mode B - {title} Duration per User, compared across Folders")
        ax.set_ylabel("Duration (s)")
        ax.set_xlabel("User")
        plt.grid(axis="y", alpha=0.3)
        plt.legend(title="Folder")
        plt.tight_layout()
        plt.show()

    # Combined chart: all metrics averaged across users, hue = scenario
    metric_cols   = ["avg_duration", "p50_duration", "p90_duration", "p95_duration", "p99_duration", "max_duration"]
    metric_labels = ["Avg",          "p50",          "p90",          "p95",          "p99",          "Max"]

    combined_rows = []
    for dim, grp in per_user_summary.groupby("dimension"):
        for col, lbl in zip(metric_cols, metric_labels):
            combined_rows.append({"dimension": dim, "Metric": lbl, "Duration": round(grp[col].mean(), 1)})
    combined_df = pd.DataFrame(combined_rows)

    plt.figure(figsize=(10, 6))
    ax = sns.barplot(data=combined_df, x="Metric", y="Duration", hue="dimension", palette="Set2", order=metric_labels)
    annotate_bars(ax)
    ax.set_title("Mode B - All Metrics Combined (avg across users per folder)")
    ax.set_ylabel("Duration (s)")
    ax.set_xlabel("Metric")
    plt.grid(axis="y", alpha=0.3)
    plt.legend(title="Scenario")
    plt.tight_layout()
    plt.show()

else:
    raise ValueError(f"Unknown OUTPUT_MODE '{OUTPUT_MODE}'. Use 'A' or 'B'.")
